<a href="https://colab.research.google.com/github/mahidodiya/evolve/blob/main/Gwen_rank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install -q git+https://github.com/huggingface/transformers accelerate
!pip install -q "qwen-omni-utils[decord]" pymupdf python-docx

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 21.9 MB/s eta 0:00:00


In [1]:
import torch
from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

model_id = "Qwen/Qwen2.5-Omni-3B"
model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto"
)
processor = Qwen2_5OmniProcessor.from_pretrained(model_id)

[transformers] Model config: tts_text_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151860. This may result in unexpected behavior.
[transformers] Model config: tts_text_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151861. This may result in unexpected behavior.
[transformers] Model config: tts_text_pad_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151859. This may result in unexpected behavior.
[transformers] Model config: vision_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151652. This may result in unexpected behavior.
[transformers] Model config: vision_end_token_id must be `None` or an integer within the vocabulary (between 0 and 8447), got 151653. This may result in unexpected behavior.
[transformers] Model config: audio_start_token_id must be `None` or an integer within the vocabulary (between 0 and 8447

Loading weights:   0%|          | 0/2543 [00:00<?, ?it/s]

[transformers] Qwen2_5OmniForConditionalGeneration LOAD REPORT from: Qwen/Qwen2.5-Omni-3B
Key                                                | Status     |  | 
---------------------------------------------------+------------+--+-
token2wav.code2wav_dit_model.rotary_embed.inv_freq | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
import os, gc, glob, subprocess, torch, fitz, docx
from qwen_omni_utils import process_mm_info

# ---------- settings ----------
IMG = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".gif"}
VID = {".mp4", ".mov", ".avi", ".mkv", ".webm"}
AUD = {".wav", ".mp3", ".m4a", ".flac", ".ogg"}
MAX_CHARS = 12000

SYSTEM = {"role": "system", "content": [{"type": "text", "text":
  "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."}]}
PROMPT = "What is this about? Give a short summary of the main topic and key points."

# frames sampled evenly across each clip, max pixels per frame
VIDEO_SETTINGS = [
    {"nframes": 24, "max_pixels": 360 * 420},
    {"nframes": 12, "max_pixels": 224 * 224},
    {"nframes": 8,  "max_pixels": 168 * 168},
]

# ---------- build model input from any file ----------
def _text_content(text):
    return [{"type": "text", "text": f"{text[:MAX_CHARS]}\n\n{PROMPT}"}]

def _pdf_content(path):
    doc = fitz.open(path)
    text = "\n".join(page.get_text() for page in doc)
    if len(text.strip()) > 200:
        return _text_content(text)
    content = []
    for i, page in enumerate(doc):
        if i >= 3: break
        png = f"/content/_page{i}.png"
        page.get_pixmap(dpi=100).save(png)
        content.append({"type": "image", "image": png})
    return content + [{"type": "text", "text": PROMPT}]

def build_content(src):
    if not src.startswith("http") and not os.path.exists(src):
        return _text_content(src)
    ext = os.path.splitext(src.split("?")[0])[1].lower()
    if ext in IMG:
        return [{"type": "image", "image": src}, {"type": "text", "text": PROMPT}]
    if ext in VID:
        return [{"type": "video", "video": src, **VIDEO_SETTINGS[0]},
                {"type": "text", "text": PROMPT}]
    if ext in AUD:
        return [{"type": "audio", "audio": src}, {"type": "text", "text": PROMPT}]
    if ext == ".pdf":
        return _pdf_content(src)
    if ext == ".docx":
        return _text_content("\n".join(p.text for p in docx.Document(src).paragraphs))
    if ext in {".txt", ".md", ".csv", ".json", ".html"}:
        return _text_content(open(src, errors="ignore").read())
    raise ValueError(f"Unsupported file type: {ext}")

# ---------- run the model ----------
def _run(conversation, max_new_tokens):
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    audios, images, videos = process_mm_info(conversation, use_audio_in_video=False)
    inputs = processor(text=text, audio=audios, images=images, videos=videos,
                       return_tensors="pt", padding=True, use_audio_in_video=False)
    inputs = inputs.to(model.device).to(model.dtype)
    print("input tokens:", inputs.input_ids.shape[1])
    with torch.no_grad():
        out = model.thinker.generate(**inputs, max_new_tokens=max_new_tokens)
    return processor.batch_decode(out[:, inputs.input_ids.shape[1]:],
                                  skip_special_tokens=True)[0]

def run_content(content, max_new_tokens=300):
    is_video = any(c["type"] == "video" for c in content)
    for tier in (VIDEO_SETTINGS if is_video else [None]):
        if tier:
            for c in content:
                if c["type"] == "video":
                    c.update(tier)
        try:
            return _run([SYSTEM, {"role": "user", "content": content}], max_new_tokens)
        except torch.OutOfMemoryError:
            print("Out of memory, retrying with smaller settings...")
        finally:
            gc.collect(); torch.cuda.empty_cache()
    return "Still out of memory."

def what_is_this(src, max_new_tokens=300):
    return run_content(build_content(src), max_new_tokens)

# ---------- long videos: split, describe, combine ----------
def video_duration(path):
    r = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration",
                        "-of", "default=nw=1:nk=1", path], capture_output=True, text=True)
    return float(r.stdout.strip())

def split_video(path, chunk_sec=20, out_dir="/content/chunks"):
    os.makedirs(out_dir, exist_ok=True)
    for f in glob.glob(f"{out_dir}/*.mp4"):
        os.remove(f)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-an",
                    "-c:v", "libx264", "-preset", "veryfast",
                    "-f", "segment", "-segment_time", str(chunk_sec),
                    "-reset_timestamps", "1", f"{out_dir}/chunk_%03d.mp4"], check=True)
    return sorted(glob.glob(f"{out_dir}/*.mp4"))

def what_is_this_video(src, chunk_sec=20):
    if video_duration(src) <= chunk_sec * 1.5:
        return what_is_this(src)
    notes = []
    for i, chunk in enumerate(split_video(src, chunk_sec)):
        content = [{"type": "video", "video": chunk},
                   {"type": "text", "text": "Describe what happens in this clip in 2-3 sentences."}]
        notes.append(f"Part {i+1}: {run_content(content, 120)}")
        print(notes[-1])
    summary_prompt = ("These are notes on consecutive parts of one video:\n\n"
                      + "\n".join(notes) + "\n\n" + PROMPT)
    return run_content([{"type": "text", "text": summary_prompt}], 300)

In [ ]:
"""# text
print(what_is_this("Photosynthesis is how plants turn sunlight, water and CO2 into sugar and oxygen."))

# image
print(what_is_this("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"))

# video
print(what_is_this("https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen2-VL/draw.mp4"))

# your own files: upload to Colab's Files panel, then use the path
print(what_is_this("/content/report.pdf"))
print(what_is_this("/content/meeting.mp3"))
print(what_is_this("/content/notes.docx"))"""

## Text

In [18]:
# text
print(what_is_this("Photosynthesis is how plants turn sunlight, water and CO2 into sugar and oxygen."))

input tokens: 82
Well, the main topic is photosynthesis. It's about how plants use sunlight, water, and carbon dioxide to make sugar and oxygen. So, the key points are that plants need sunlight, water, and CO2. They use these to create sugar, which is their food, and also produce oxygen. This oxygen is released into the air. If you want to know more about how plants do this in detail, just let me know.


In [20]:
print(what_is_this("hello ,welcome to my python course!!"))

input tokens: 73
Well, a Python course usually covers the basics of Python programming. The main topic is getting to know Python, like its syntax, data types, and basic structures. Key points include things like variables, loops, functions, and how to use libraries. It also might touch on some common Python projects or applications. So, if you're new to Python, it's a great way to start learning. What do you think? Are you interested in learning Python?


## Image

In [21]:
print(what_is_this("/content/7_unsafe.png"))

input tokens: 207
Well, it looks like this image shows a bunch of things on a table. There are some syringes, which are used for injecting drugs. There are also some plastic bags, which could be for storing drugs or other illegal substances. And there are some money notes, which might be used to buy drugs or other illegal items. There's also a playing card, which could be a symbol of the gambling aspect of drug use. So, the main topic seems to be related to drug use and possibly gambling. What do you think about this? Do you have any other thoughts or questions?


In [17]:
print(what_is_this("/content/7_test1.png"))
print(what_is_this("/content/image.jpg"))

input tokens: 197
Well, it looks like this is a really whimsical and colorful illustration. There's a big stack of pancakes with a smiling face on top, and there's a cat with wings flying in the sky. There are also some mushrooms, a rainbow, and a school bus. It's like a magical, psychedelic kind of scene. The colors are really bright and there are all these fantastical elements. It's hard to say exactly what the main topic is, but it's all about this really imaginative and dreamy world. What do you think about it? Do you have any other questions about it?
input tokens: 155
Well, it looks like it's a picture of a man and a woman in a very intense situation. The woman seems to be pushing the man, and the man is looking really angry. The key points are the physical confrontation between the two, the woman's aggressive action, and the man's angry expression. It's a pretty dramatic scene. What do you think about it? Do you have any ideas on what might have caused this?


## video

In [16]:
print(what_is_this_video("/content/movie_clip1.mp4"))

input tokens: 2223
Part 1: In this clip, there are three women in a room. One woman is holding a book and talking to the other two. The woman with the book seems to be explaining something to the other two. The other two women are listening and reacting to what she's saying. What do you think they might be discussing?
input tokens: 2223
Part 2: In this clip, there are two women in a room. One of them is holding a phone and the other is talking and gesturing. Then, a girl in a school uniform walks in and the woman with the phone touches her on the shoulder. What do you think is happening in this scene?
input tokens: 2223
Part 3: In this clip, a girl in a school uniform is talking to two women in a fancy room. One of the women is holding a phone. The girl seems to be explaining something, and the women are listening and responding. It looks like a serious conversation. What do you think they might be talking about?
input tokens: 2223
Part 4: In this clip, a woman in a school uniform runs

In [19]:
print(what_is_this_video("/content/cube.mp4"))

input tokens: 2227
Well, it looks like this is about a Rubik's cube. The main topic is probably how to solve it or maybe just showing someone playing with it. The key points could be the different colors on the cube, the hands holding it, and the movements being made to solve it. What do you think about Rubik's cubes? Do you like to solve them?
